In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False):
        super(SeparableConv2d, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size, 
            stride=stride, padding=padding, groups=in_channels, bias=bias
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, 
            stride=1, padding=0, bias=bias
        )
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

In [ ]:
class EntryFlow(nn.Module):
    def __init__(self):
        super(EntryFlow, self).__init__()
        

        self.conv1 = nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=False)
        

        self.conv2 = nn.Conv2d(32, 64, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)

        self.block1_sep1 = SeparableConv2d(64, 128, 3)
        self.block1_bn1 = nn.BatchNorm2d(128)
        
        self.block1_sep2 = SeparableConv2d(128, 128, 3)
        self.block1_bn2 = nn.BatchNorm2d(128)
        
        self.block1_pool = nn.MaxPool2d(3, stride=2, padding=1)

        self.block1_skip = nn.Conv2d(64, 128, 1, stride=2, bias=False)
        self.block1_skip_bn = nn.BatchNorm2d(128)

        self.block2_sep1 = SeparableConv2d(128, 256, 3)
        self.block2_bn1 = nn.BatchNorm2d(256)
        
        self.block2_sep2 = SeparableConv2d(256, 256, 3)
        self.block2_bn2 = nn.BatchNorm2d(256)
        
        self.block2_pool = nn.MaxPool2d(3, stride=2, padding=1)

        self.block2_skip = nn.Conv2d(128, 256, 1, stride=2, bias=False)
        self.block2_skip_bn = nn.BatchNorm2d(256)

        self.block3_sep1 = SeparableConv2d(256, 728, 3)
        self.block3_bn1 = nn.BatchNorm2d(728)
        
        self.block3_sep2 = SeparableConv2d(728, 728, 3)
        self.block3_bn2 = nn.BatchNorm2d(728)
        
        self.block3_pool = nn.MaxPool2d(3, stride=2, padding=1)

        self.block3_skip = nn.Conv2d(256, 728, 1, stride=2, bias=False)
        self.block3_skip_bn = nn.BatchNorm2d(728)

    def forward(self, x):

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        

        residual = self.block1_skip_bn(self.block1_skip(x))
        
        out = self.block1_sep1(x)
        out = self.block1_bn1(out)
        out = self.relu(out)
        out = self.block1_sep2(out)
        out = self.block1_bn2(out)
        out = self.block1_pool(out)
        
        x = out + residual

        residual = self.block2_skip_bn(self.block2_skip(x))
        
        out = self.relu(x)
        out = self.block2_sep1(out)
        out = self.block2_bn1(out)
        out = self.relu(out)
        out = self.block2_sep2(out)
        out = self.block2_bn2(out)
        out = self.block2_pool(out)
        
        x = out + residual

        residual = self.block3_skip_bn(self.block3_skip(x))
        
        out = self.relu(x)
        out = self.block3_sep1(out)
        out = self.block3_bn1(out)
        out = self.relu(out)
        out = self.block3_sep2(out)
        out = self.block3_bn2(out)
        out = self.block3_pool(out)
        
        x = out + residual
        
        return x

In [ ]:
class MiddleFlowBlock(nn.Module):
    def __init__(self):
        super(MiddleFlowBlock, self).__init__()
        self.relu = nn.ReLU(inplace=False)
        
        self.sep1 = SeparableConv2d(728, 728, 3)
        self.bn1 = nn.BatchNorm2d(728)
        
        self.sep2 = SeparableConv2d(728, 728, 3)
        self.bn2 = nn.BatchNorm2d(728)
        
        self.sep3 = SeparableConv2d(728, 728, 3)
        self.bn3 = nn.BatchNorm2d(728)
    
    def forward(self, x):
        residual = x

        out = self.relu(x)
        out = self.sep1(out)
        out = self.bn1(out)
        
        out = self.relu(out)
        out = self.sep2(out)
        out = self.bn2(out)
        
        out = self.relu(out)
        out = self.sep3(out)
        out = self.bn3(out)
        
        return out + residual

class MiddleFlow(nn.Module):
    def __init__(self):
        super(MiddleFlow, self).__init__()
        self.blocks = nn.Sequential(*[MiddleFlowBlock() for _ in range(8)])

    def forward(self, x):
        return self.blocks(x)

In [ ]:
class ExitFlow(nn.Module):
    def __init__(self):
        super(ExitFlow, self).__init__()
        self.relu = nn.ReLU(inplace=False)

        self.sep1 = SeparableConv2d(728, 728, 3)
        self.bn1 = nn.BatchNorm2d(728)
        
        self.sep2 = SeparableConv2d(728, 1024, 3)
        self.bn2 = nn.BatchNorm2d(1024)
        
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)
        
        self.skip = nn.Conv2d(728, 1024, 1, stride=2, bias=False)
        self.skip_bn = nn.BatchNorm2d(1024)

        self.sep3 = SeparableConv2d(1024, 1536, 3)
        self.bn3 = nn.BatchNorm2d(1536)
        
        self.sep4 = SeparableConv2d(1536, 2048, 3)
        self.bn4 = nn.BatchNorm2d(2048)

    def forward(self, x):

        residual = self.skip_bn(self.skip(x))
        
        out = self.relu(x)
        out = self.sep1(out)
        out = self.bn1(out)
        
        out = self.relu(out)
        out = self.sep2(out)
        out = self.bn2(out)
        
        out = self.pool(out)
        
        x = out + residual
        

        x = self.sep3(x)
        x = self.bn3(x)
        x = self.relu(x)
        
        x = self.sep4(x)
        x = self.bn4(x)
        x = self.relu(x)
        
        return x

In [ ]:
class Xception(nn.Module):
    def __init__(self, num_classes=2):
        super(Xception, self).__init__()
        
        self.entry_flow = EntryFlow()
        self.middle_flow = MiddleFlow()
        self.exit_flow = ExitFlow()
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(2048, num_classes)

        self._initialize_weights()

    def forward(self, x):
        x = self.entry_flow(x)
        x = self.middle_flow(x)
        x = self.exit_flow(x)
        
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [ ]:
import pandas as pd

csv_path = "/kaggle/input/deepfakedata/dataset/train_solution.csv"
df = pd.read_csv(csv_path).iloc[:, :2].copy()
df.columns = ["image_id", "label"]
df["image_id"] = df["image_id"].astype(int)
df["label"] = df["label"].astype(int)

df.head()


In [ ]:
df

In [ ]:
import torch
import pandas as pd

def calculate_class_weights(csv_path):

    df = pd.read_csv(csv_path).iloc[:, :2].copy()
    df.columns = ["image_id", "label"]
    labels = df["label"].values.astype(int)

    class_count = [int((labels == 0).sum()), int((labels == 1).sum())]
    n_samples = sum(class_count)

    weights = [n_samples / (2 * max(1, class_count[i])) for i in range(len(class_count))]

    return torch.tensor(weights, dtype=torch.float)

csv_path = '/kaggle/input/deepfakedata/dataset/train_solution.csv'
weights = calculate_class_weights(csv_path)
criterion = nn.CrossEntropyLoss()
print("Class weights:", weights)


In [ ]:
import torchvision.transforms as transforms

data_transforms = {
    'train': transforms.Compose([
        #transforms.Resize((299, 299)),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(degrees=15),


        transforms.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.05, hue=0),

        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.1),

        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),


        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ]),


    'val': transforms.Compose([
       #transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
}


In [ ]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
from PIL import Image

class DeepfakeDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file).iloc[:, :2].copy()
        self.annotations.columns = ['image_id', 'label']
        self.annotations['image_id'] = self.annotations['image_id'].astype(int)
        self.annotations['label'] = self.annotations['label'].astype(int)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()


        img_id = self.annotations.iloc[idx, 0]
        img_name = os.path.join(self.root_dir, f"{img_id}.jpg")
        image = Image.open(img_name).convert('RGB')
        label = self.annotations.iloc[idx, 1]
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torch.utils.data import DataLoader

dataset = DeepfakeDataset(csv_file='/kaggle/input/deepfakedata/dataset/train_solution.csv', root_dir='/kaggle/input/deepfakedata/dataset/train_images',
                          transform=data_transforms['train'])

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler


csv_path = '/kaggle/input/deepfakedata/dataset/train_solution.csv'
df = pd.read_csv(csv_path).iloc[:, :2].copy()
df.columns = ['image_id', 'label']
df['image_id'] = df['image_id'].astype(int)
df['label'] = df['label'].astype(int)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Xception(num_classes=2)


if torch.cuda.device_count() > 1:
    print(f"{torch.cuda.device_count()} GPU")
    model = nn.DataParallel(model)
else:
    print(f"{device}")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0005, betas=(0.9, 0.999), eps=1e-08)

scheduler = StepLR(optimizer, step_size=17, gamma=0.1)

train_dataset = DeepfakeDataset(csv_file='train.csv',
                                root_dir='/kaggle/input/deepfakedata/dataset/train_images',
                                transform=data_transforms['train'])
val_dataset = DeepfakeDataset(csv_file='val.csv',
                              root_dir='/kaggle/input/deepfakedata/dataset/train_images',
                              transform=data_transforms['val'])


y = train_df['label'].values.astype(int)
class_counts = np.bincount(y)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[y]
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_weights).double(),
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


train_labels = train_dataset.annotations['label'].values.astype(int)
class_count = [int((train_labels == 0).sum()), int((train_labels == 1).sum())]
n_samples = sum(class_count)
w = [n_samples / (2 * max(1, x)) for x in class_count]
class_weights_t = torch.FloatTensor(w).to(device)
#criterion.weight = class_weights_t
print(f"Class weights for WCE: {class_weights_t}")


In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from tqdm import tqdm

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []


    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

    return epoch_loss, epoch_acc, f1, precision, recall

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_probs = []
    all_labels = []

    with torch.no_grad():

        progress_bar = tqdm(dataloader, desc="Validating", leave=False)

        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

            progress_bar.set_postfix({'val_loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(dataloader.dataset)


    probs_np = np.asarray(all_probs, dtype=np.float32)
    labels_np = np.asarray(all_labels, dtype=np.int64)

    thresholds = np.linspace(0.01, 0.99, 99)
    best_f1, best_thr = -1.0, 0.5
    best_preds = None

    for thr in thresholds:
        preds = (probs_np >= thr).astype(np.int64)
        _, _, f1, _ = precision_recall_fscore_support(labels_np, preds, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_thr = float(thr)
            best_preds = preds


    epoch_acc = accuracy_score(labels_np, best_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels_np, best_preds, average='binary', zero_division=0)
    cm = confusion_matrix(labels_np, best_preds)

    return epoch_loss, epoch_acc, f1, precision, recall, cm, best_thr


In [ ]:
import torch
import matplotlib.pyplot as plt
import os
from tqdm import trange


load_path = '/kaggle/input/mlyandexmodel1/models/best_xception_model.pth'
resume_training = False

if resume_training and os.path.exists(load_path):
    try:
        checkpoint = torch.load(load_path, map_location=device)
        state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) else checkpoint
        

        from collections import OrderedDict
        new_state_dict = OrderedDict()
        
        is_model_parallel = isinstance(model, nn.DataParallel)
        
        for k, v in state_dict.items():
            if k.startswith('module.') and not is_model_parallel:
                name = k[7:]
            elif not k.startswith('module.') and is_model_parallel:
                name = 'module.' + k
            else:
                name = k
            new_state_dict[name] = v
            
        model.load_state_dict(new_state_dict)

    except Exception as e:
        pass


os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

num_epochs = 50
best_val_f1 = -1.0
best_threshold = 0.0


train_f1_history = []
val_f1_history = []
train_loss_history = []
val_loss_history = []

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    train_loss, train_acc, train_f1, train_precision, train_recall = train_epoch(
        model, train_loader, criterion, optimizer, device)

    val_loss, val_acc, val_f1, val_precision, val_recall, cm, val_thr = validate(
        model, val_loader, criterion, device)

    print(f'Best threshold on val this epoch: {val_thr:.2f}')

    train_f1_history.append(train_f1)
    val_f1_history.append(val_f1)
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}')
    print(f'Confusion Matrix:\n{cm}')

    scheduler.step()

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_threshold = val_thr
        best_model_path = 'models/best_xception_model.pth'

        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save({'model_state_dict': model_to_save.state_dict(), 'threshold': val_thr}, best_model_path)
        print(f'New best model saved with Val F1: {val_f1:.4f} -> {best_model_path}')


    torch.save({'model_state_dict': model_to_save.state_dict(), 'threshold': val_thr}, 'models/last_xception_model.pth')

    if (epoch + 1) % 1 == 0 or epoch == num_epochs - 1:
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.plot(train_f1_history, label='Train F1', marker='o')
        plt.plot(val_f1_history, label='Val F1', marker='o')
        plt.title('F1-Score History')
        plt.xlabel('Epoch')
        plt.ylabel('F1-Score')
        plt.legend()
        plt.grid(True)


        plt.subplot(1, 2, 2)
        plt.plot(train_loss_history, label='Train Loss', marker='o')
        plt.plot(val_loss_history, label='Val Loss', marker='o')
        plt.title('Loss History')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plot_path = f'plots/training_progress_epoch_{epoch+1}.png'
        plt.savefig(plot_path)
        plt.close()
        print(f'📈 Plots saved to {plot_path}')

print('Training complete.')
print(f'Best validation F1: {best_val_f1:.4f}')
print(f'Best threshold: {best_threshold:.2f}')

In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


TEST_DIR = '/kaggle/input/deepfakedata/dataset/test_images' 
MODEL_PATH = 'models/best_xception_model.pth'
OUTPUT_CSV = 'submission.csv'


test_transform = data_transforms['val']


class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.files = [f for f in os.listdir(root_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        img_path = os.path.join(self.root_dir, filename)

        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)

        image_id = int(os.path.splitext(filename)[0])
        
        return image, image_id


test_dataset = TestDataset(root_dir=TEST_DIR, transform=test_transform)


test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Xception(num_classes=2).to(device)

if os.path.exists(MODEL_PATH):

    checkpoint = torch.load(MODEL_PATH, map_location=device)
    

    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) else checkpoint
    

    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v
    
    model.load_state_dict(new_state_dict)
    

    threshold = checkpoint.get('threshold', 0.5) if isinstance(checkpoint, dict) else 0.5
else:
    threshold = 0.5


if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

model.eval()


all_ids = []
all_preds = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Inference"):
        images = images.to(device)

        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        

        preds = (probs >= threshold).long().cpu().numpy()
        
        all_ids.extend(ids.numpy())
        all_preds.extend(preds)


submission_df = pd.DataFrame({
    'id': all_ids,
    'target_feature': all_preds
})


submission_df = submission_df.sort_values(by='id').reset_index(drop=True)


submission_df.to_csv(OUTPUT_CSV, index=False)

print(submission_df.head())